In [13]:
import pandas as pd
import os
import joblib
from gene_thesaurus import GeneThesaurus

In [14]:
MODELS_PATH = "/home/mariya/Data/pallas/models/gex"
model_path = os.path.join(MODELS_PATH, "rf_model.pkl")
selected_features_path = os.path.join(MODELS_PATH, "selected_features.txt")

DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"

TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-gex")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.gex.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

CALIBRATE_DATASET_PATH = os.path.join(DATASETS_PATH, "calibrate-gex")
X_calibrate_path = os.path.join(CALIBRATE_DATASET_PATH, "features.gex.csv")
y_calibrate_path = os.path.join(CALIBRATE_DATASET_PATH, "labels.csv")

TEST_DATASET_PATH = os.path.join(DATASETS_PATH, "test-gex")
X_test_path = os.path.join(TEST_DATASET_PATH, "features.gex.csv")
y_test_path = os.path.join(TEST_DATASET_PATH, "labels.csv")

model = joblib.load(model_path)
selected_features = pd.read_csv(selected_features_path, header=None)[0].tolist()

In [ ]:
# Merge test datasets and save
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, index_col=0)
X_cal = pd.read_csv(X_calibrate_path, index_col=0)
y_cal = pd.read_csv(y_calibrate_path, index_col=0)

# Append X_cal to X_test and y_cal to y_test
X_merged = pd.concat([X_cal, X_test], axis=0)
y_merged = pd.concat([y_cal, y_test], axis=0)

# Subset merged datasets to selected features
X_merged = X_merged[selected_features]

# Align indices
X_merged = X_merged.loc[y_merged.index]

# Save merged datasets
X_merged.to_csv(os.path.join(MODELS_PATH, "merged.test.cal.features.csv"))
y_merged.to_csv(os.path.join(MODELS_PATH, "merged.test.cal.labels.csv"))

In [16]:
importances = model.feature_importances_

# Rank cpg sites by importance
importances_df = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(importances_df["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
importances_df["Gene symbol"] = importances_df["Gene"].map(translated)

print(importances_df.shape)

print("\nTop 20 genes driving classification (final model):")
print(importances_df.head(20))

(3923, 3)

Top 20 genes driving classification (final model):
                 Gene  Importance Gene symbol
3067  ENSG00000184507    0.029388       NUTM1
317   ENSG00000079102    0.025635     RUNX1T1
1730  ENSG00000145794    0.019836      MEGF10
493   ENSG00000101115    0.017745       SALL4
3632  ENSG00000248329    0.016787       APELA
3699  ENSG00000259803    0.012556    SLC22A31
2267  ENSG00000164100    0.011442       NDST3
2025  ENSG00000157680    0.011384        DGKI
626   ENSG00000105642    0.011167       KCNN1
3081  ENSG00000185046    0.011096      ANKS1B
715   ENSG00000108511    0.010603       HOXB6
1071  ENSG00000124302    0.010031       CHST8
3352  ENSG00000197705    0.009977      KLHL14
972   ENSG00000120075    0.008896       HOXB5
2455  ENSG00000168004    0.008786      PLAAT5
1668  ENSG00000143995    0.008637       MEIS1
909   ENSG00000116703    0.008221         PDC
1104  ENSG00000125821    0.007942        DTD1
1657  ENSG00000143816    0.007469       WNT9A
327   ENSG00000080

In [17]:
# Filter out all importances that are zero
non_zero_importances_df = importances_df[importances_df["Importance"] > 0]
print(f"\nNumber of genes with non-zero importance: {non_zero_importances_df.shape[0]}")


Number of genes with non-zero importance: 3923


In [18]:
# Save the ranked non-zero importances to a CSV file for further analysis
output_path = os.path.join(MODELS_PATH, "gene_importances.csv")
non_zero_importances_df.to_csv(output_path, index=False)
print(f"\nRanked gene importances saved to: {output_path}")

# Also save just the features names with non-zero importance
# as a txt file of the same format as selected_features.txt
# Name it selected_features.filtered.sorted.txt
filtered_sorted_features_path = os.path.join(MODELS_PATH, "selected_features.filtered.sorted.txt")
non_zero_importances_df[non_zero_importances_df["Importance"] > 0]["Gene"].to_csv(filtered_sorted_features_path, index=False, header=False)
print(f"\nFiltered and sorted feature names saved to: {filtered_sorted_features_path}")


Ranked gene importances saved to: /home/mariya/Data/pallas/models/gex/gene_importances.csv

Filtered and sorted feature names saved to: /home/mariya/Data/pallas/models/gex/selected_features.filtered.sorted.txt


In [19]:
# Save the first 500 and 1000 features with non-zero importance to separate txt files
top_500_features_path = os.path.join(MODELS_PATH, "selected_features.top_500.txt")
top_1000_features_path = os.path.join(MODELS_PATH, "selected_features.top_1000.txt")
non_zero_importances_df.head(500)["Gene"].to_csv(top_500_features_path, index=False, header=False)
non_zero_importances_df.head(1000)["Gene"].to_csv(top_1000_features_path, index=False, header=False)
print(f"\nTop 500 feature names saved to: {top_500_features_path}")
print(f"Top 1000 feature names saved to: {top_1000_features_path}")


Top 500 feature names saved to: /home/mariya/Data/pallas/models/gex/selected_features.top_500.txt
Top 1000 feature names saved to: /home/mariya/Data/pallas/models/gex/selected_features.top_1000.txt
